# RaccoonBot OpenVLA 멀티태스크 학습 노트북

이 노트북은 현재 저장소의 `raccoon_grasp`, `raccoon_push`,
`raccoon_pick_and_place` TFDS를 동일 가중치로 사용하는
`raccoon_task_balanced` mixture 학습 절차입니다.


## 0. 프로젝트 경로 설정

노트북을 저장소 루트에서 열면 자동으로 경로를 찾습니다. 다른 위치에서
열었다면 `RACCOONBOT_ROOT` 환경 변수에 clone 경로를 지정합니다.


In [ ]:
import os
from pathlib import Path

start = Path(os.environ.get("RACCOONBOT_ROOT", Path.cwd())).expanduser().resolve()
candidates = [start, start / "Raccoonbot_Openvla", start / "openVLA-with-Raccoonbot"]
PROJECT_ROOT = next((p for p in candidates if (p / "openvla").is_dir() and (p / "Mujoco").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "저장소를 찾지 못했습니다. RACCOONBOT_ROOT=/path/to/repository 를 설정하세요."
    )

OPENVLA_DIR = PROJECT_ROOT / "openvla"
TFDS_DIR = PROJECT_ROOT / "tensorflow_datasets"
print("PROJECT_ROOT =", PROJECT_ROOT)
print("OPENVLA_DIR  =", OPENVLA_DIR)
print("TFDS_DIR     =", TFDS_DIR)


## 1. 환경 설치 및 확인


In [ ]:
os.chdir(OPENVLA_DIR)
!pip install .
!python3 -c "import torch; print('torch=', torch.__version__, 'cuda=', torch.cuda.is_available())"
!python3 -c "import tensorflow as tf; print('tensorflow=', tf.__version__)"
!nvidia-smi


## 2. MuJoCo demonstration 생성

이미 raw 데이터와 TFDS가 준비되어 있으면 이 단계는 건너뜁니다.


In [ ]:
os.chdir(PROJECT_ROOT / "Mujoco")
# 잡기 데이터: 기본 1,200개
!python3 raccoon_grasp_multicolor_scene_dataset.py

# 밀기 + 집어서 옮기기 데이터: 기본 720개
!python3 raccoon_multitask_colored_objects_dataset.py


## 3. task-balanced intermediate 변환


In [ ]:
os.chdir(PROJECT_ROOT / "Mujoco" / "raccoon_dataset")
!python3 prepare_task_balanced_intermediate.py


## 4. TFDS 빌드

최종 학습 데이터셋 이름은 **`raccoon_task_balanced`** 입니다. 이 이름은
세 TFDS를 1:1:1 가중치로 샘플링하는 OpenVLA mixture 이름입니다.


In [ ]:
os.chdir(PROJECT_ROOT / "Mujoco" / "rlds_dataset_builder")
!python3 build_task_balanced_tfds.py --overwrite

for name in ["raccoon_grasp", "raccoon_push", "raccoon_pick_and_place"]:
    path = TFDS_DIR / name / "1.0.0" / "dataset_info.json"
    print(name, "OK" if path.is_file() else "MISSING", path)


## 5. OpenVLA LoRA 학습

중요 사항:

- `vla-scripts/finetune.py`는 반드시 `openvla/` 디렉터리에서 실행합니다.
- `--dataset_name`은 **`raccoon_task_balanced`** 입니다.
- 색상 학습 baseline은 `--image_aug False`를 사용합니다.
- 아래 설정은 2,000 step 학습, 500 step 간격 저장입니다.


In [ ]:
os.chdir(OPENVLA_DIR)
os.environ["PYTHONPATH"] = f"{OPENVLA_DIR}:{os.environ.get('PYTHONPATH', '')}"
print("working directory =", Path.cwd())
print("finetune script exists =", (OPENVLA_DIR / "vla-scripts" / "finetune.py").is_file())
print("TFDS exists =", TFDS_DIR.is_dir())


In [ ]:
!WANDB_MODE=disabled CUDA_VISIBLE_DEVICES=0 \
torchrun --standalone --nnodes 1 --nproc-per-node 1 vla-scripts/finetune.py \
  --vla_path openvla/openvla-7b \
  --data_root_dir "{TFDS_DIR}" \
  --dataset_name raccoon_task_balanced \
  --run_root_dir "{OPENVLA_DIR / 'openvla-runs'}" \
  --adapter_tmp_dir "{OPENVLA_DIR / 'openvla-adapter-tmp'}" \
  --lora_rank 32 \
  --batch_size 8 \
  --grad_accumulation_steps 2 \
  --learning_rate 2e-4 \
  --max_steps 2000 \
  --save_steps 500 \
  --image_aug False \
  --run_id_note raccoon-task-balanced


## 6. 학습 상태 및 결과 경로 확인


In [ ]:
!ps aux | grep '[f]inetune.py' || true
!nvidia-smi

RUN_DIR = OPENVLA_DIR / "openvla-runs" / (
    "openvla-7b+raccoon_task_balanced+b16+lr-0.0002+"
    "lora-r32+dropout-0.0--raccoon-task-balanced"
)
print("예상 checkpoint 경로 =", RUN_DIR)
print("존재 여부 =", RUN_DIR.is_dir())


## 7. 학습된 checkpoint 추론 서버 실행

학습 완료 후 아래 셀을 실행합니다. normalization key는 checkpoint의 `dataset_statistics.json`에서 확인합니다.
혼합 데이터 학습 결과에는 작업별 key가 저장되므로 잡기 추론은
`raccoon_grasp`, 밀기는 `raccoon_push`, 집어서 옮기기는
`raccoon_pick_and_place`를 사용합니다.


In [ ]:
os.chdir(OPENVLA_DIR)
!CUDA_VISIBLE_DEVICES=0 python3 openvla_server.py \
  --model_path "{RUN_DIR}" \
  --default-unnorm-key raccoon_grasp \
  --host 0.0.0.0 \
  --port 8000 \
  --device cuda
